#  Model D — Distractor Generation

##  1. Import Libraries

In [ ]:
import os
import string
import warnings

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
warnings.filterwarnings("ignore")

##  2. Load & Clean Dataset

In [11]:
train_df = pd.read_csv("../data/raw/train.csv")
train_df = train_df.dropna(subset=["article", "A", "B", "C", "D"])

def clean_text(text):
    """Lowercase, strip punctuation, and remove stopwords from a string."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = [w for w in text.split() if w not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)

print(f" Dataset loaded — {len(train_df):,} samples ready")

 Dataset loaded — 87,852 samples ready


##  3. Helper: Extract Correct Answer Text


In [12]:
def get_correct_answer_text(row):
    """Return the cleaned text of the correct answer option."""
    correct_option_label = row['answer'] # 'A', 'B', 'C', or 'D'
    return clean_text(row[correct_option_label])

##  4. Core Function: Generate Distractors

In [13]:
def generate_distractors(article, correct_answer_text, top_n=3):
    """
    Generate  distractor options for a given article and correct answer.

    Strategy:
      - Extract unique alpha tokens longer than 4 chars from the article
      - Exclude tokens already present in the correct answer
      - Rank remaining candidates by TF-IDF cosine similarity to the correct answer
      - Return the most similar (but not identical) candidates as distractors
    """

    # --- Candidate extraction ---
    words = str(article).split()
    candidates = list(set([
        word for word in words
        if len(word) > 4
        and word.isalpha()
        and word.lower() not in str(correct_answer_text).lower()
    ]))

    # Fallback when the article has too few usable words
    if len(candidates) < top_n:
        return ["none of the above", "all of the above", "cannot be determined"][:top_n]

    try:
        # --- TF-IDF vectorisation ---
        corpus = [str(correct_answer_text)] + candidates
        tfidf_matrix = TfidfVectorizer().fit_transform(corpus)

        # --- Rank by cosine similarity (highest first) ---
        similarities = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
        ranked_indices = np.argsort(similarities)[::-1]

        # Collect unique top-N distractors
        distractors = []
        for idx in ranked_indices:
            candidate = candidates[idx]
            if candidate not in distractors:
                distractors.append(candidate)
            if len(distractors) == top_n:
                break

        # Pad with placeholder if still short
        while len(distractors) < top_n:
            distractors.append("unknown")

        return distractors

    except Exception:
        return ["Option X", "Option Y", "Option Z"]


##  5. Run Distractor Generation Pipeline


In [14]:
 
sample_df = train_df 
results = []

print(f"Generating distractors for all {len(sample_df):,} questions. This will take a while...")

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    correct_answer = get_correct_answer_text(row)
    
    clean_article = clean_text(row["article"])
    clean_question = clean_text(row["question"])
    
    distractors = generate_distractors(clean_article, correct_answer)

    results.append({
        "Question":       clean_question,
        "Correct_Answer": correct_answer,
        "Distractor_1":   distractors[0],
        "Distractor_2":   distractors[1],
        "Distractor_3":   distractors[2],
    })

results_df = pd.DataFrame(results)
print(f"\nGenerated distractors for {len(results_df):,} questions")
results_df.head()

Generating distractors for all 87,852 questions. This will take a while...


  0%|          | 0/87852 [00:00<?, ?it/s]

100%|██████████| 87852/87852 [02:56<00:00, 496.97it/s]



Generated distractors for 87,852 questions


,Question,Correct_Answer,Distractor_1,Distractor_2,Distractor_3
0,writer came high school summer camphe n,student,summer,college,leading
1,times did writer invite boy join activities,twice,summer,college,leading
2,bumpkin thought,short fat woman changed young girl room,slowly,small,light
3,room saw,lift,slowly,brought,stepped
4,bumpkin knowledge,modern city life,slowly,brought,stepped


##  6. Evaluate Distractor Quality


In [17]:
# Unique distractor vocabulary across all generated options
all_distractors = (
    results_df["Distractor_1"].tolist() +
    results_df["Distractor_2"].tolist() +
    results_df["Distractor_3"].tolist()
)
unique_count = len(set(all_distractors))

# Average word length of first distractors as a proxy for meaningfulness
avg_length = np.mean([len(str(x).split()) for x in results_df["Distractor_1"]])

print(f" Unique distractors generated : {unique_count}")
print(f" Average distractor word length : {avg_length:.2f}")

 Unique distractors generated : 8064
 Average distractor word length : 1.00


##  7. Save Results


In [18]:
output_path = "../reports/distractor_generation_results.csv"
os.makedirs("../reports", exist_ok=True)

results_df.to_csv(output_path, index=False)
print(f" Results saved to: {output_path}")

 Results saved to: ../reports/distractor_generation_results.csv
